# Load and Pre-Process the Raw Data Files

This is being done in a notebook to better tailor the individual pre-processing steps, of each data file, to its individual needs. 

In [2]:
# Import libraries
import pandas as pd

## 1. Read the Product Data

The product data consists of these columns:
1. Labelgruppe_norm	- Brand of the product
2. Geschlecht - Gender of target group
3. Altersgruppe	- Age group
4. LiefArtNr - Supplier article number
5. LiefFarbe - Supplier colour
6. WGR_norm - Product category (normalized)
7. Kategorie - Product category (broad)
8. Bild_URL_1 - URL of the image
9. Bild_URL_2	
10. Bild_URL_3	
11. Bild_URL_4	
12. Bild_URL_5

In [3]:
# Read product data
product_data = pd.read_csv('../../data/raw_data/Attribute_Bilder_MV_20241219.txt', sep=";", encoding='latin1')
product_data.head(3)

,Labelgruppe_norm,Geschlecht,Altersgruppe,LiefArtNr,LiefFarbe,WGR_norm,Kategorie,Bild_URL_1,Bild_URL_2,Bild_URL_3,Bild_URL_4,Bild_URL_5
0,zero,D,E,1063671,9706 Silver Melange,110201040010,10101,https://assets.bettybarclay.com/media/image/6a...,NaN,NaN,NaN,NaN
1,zero,D,E,1063671,Silver Melange,110201040010,10101,https://assets.bettybarclay.com/media/image/6a...,NaN,NaN,NaN,NaN
2,zero,D,E,1063821,7400 Light Sand,110201040010,10101,https://assets.bettybarclay.com/media/image/e6...,NaN,NaN,NaN,NaN


## 2. Read Attributes

The attributes data consists of the following columns:
1. Attribut Id - Broad attribute category
2. Identifier - Specific sub-attribute category
3. Beschreibung - Description of the specific sub-attribute category

In [3]:
# Read attribute data
attribute_data = pd.read_excel('../../data/raw_data/Exp. Kategorie Attribute 18Dez24 LV.xlsx', sheet_name='Wertemenge je Attribut', usecols='A:C').dropna()
attribute_data.head(3)

,Attribut Id,Identifier,Beschreibung
1,handschuheArt,fleecehandschuhe,Fleecehandschuhe
2,handschuheArt,skihandschuhe,Skihandschuhe
3,handschuheArt,fausthandschuhe,Fausthandschuhe


## 3. Read and Pre-Process Attributes + Category Data

The attribute + category data consists of the following columns:
1. Attribut Id - Broad attribute category
2. Beschreibung - Description of the specific sub-attribute category
3. Anzahl Kat - The number of product categories for this specific attribut
4. Katzuordnung - The connection between attributes and product category

In [4]:
# Read attribute category data
attribute_category_data = pd.read_excel('../../data/raw_data/Exp. Kategorie Attribute 18Dez24 LV.xlsx', sheet_name='Kat Attribut Zuordnung', usecols='A:D').dropna()

In [5]:
# Strip column 'Katzuordnung' of trailing commas and split by comma
attribute_category_data['Katzuordnung'] = attribute_category_data['Katzuordnung'].str.rstrip(',').str.split(',')

In [6]:
# Explode column 'Katzuordnung' to have one row per attribute category
attribute_category_data = attribute_category_data.explode('Katzuordnung')
attribute_category_data.head(3)

,Attribut Id,Beschreibung,Anzahl Kat,Katzuordnung
0,handschuheArt,Handschuhart,1.0,10504
1,laengeUnterteil,Unterteillänge,4.0,10203
1,laengeUnterteil,Unterteillänge,4.0,10204


## 4. Merge Attributes Data

In [7]:
attribute_merged = attribute_category_data[['Attribut Id', 'Beschreibung', 'Katzuordnung']].merge(attribute_data, left_on='Attribut Id', right_on='Attribut Id', how='left', suffixes=('_attribut', '_wertemenge'))
attribute_merged.head(3)

,Attribut Id,Beschreibung_attribut,Katzuordnung,Identifier,Beschreibung_wertemenge
0,handschuheArt,Handschuhart,10504,fleecehandschuhe,Fleecehandschuhe
1,handschuheArt,Handschuhart,10504,skihandschuhe,Skihandschuhe
2,handschuheArt,Handschuhart,10504,fausthandschuhe,Fausthandschuhe


## 5. Read the category Data

The category data set  consists of the following columns:
1. WgrNr (h+p Sprache) - Product catgeory (company internal)
2. wgrNorm - Product catgeory (normalized)
3. WgrBez - Product category description
4. Kategorie - Product category (broad)
5. Kategorie_Bezeichnung - Product category (broad) description

In [8]:
# Read category data
category_data = pd.read_excel('../../data/raw_data/Zuordnung_hpWG_zu_Klassifikation_Versand.xlsx').dropna()
category_data.head(3)

,WgrNr (h+p Sprache),wgrNorm,WgrBez,Kategorie,Kategorie_Bezeichnung
0,11-01-01-01,11010101,Herren / H-Anzüge Gesamt,1030101,Fashion / Kombiteile / Anzüge / Herren
1,11-01-01-01-0010,110101010010,Herren / H-Anzüge 2-teilig,1030101,Fashion / Kombiteile / Anzüge / Herren
2,11-01-01-01-0015,110101010015,Herren / H-Anzüge mehrteilig,1030101,Fashion / Kombiteile / Anzüge / Herren


## 6. Save Data to Preliminary Stage

In [9]:
# Save the data
product_data.to_csv('../../data/prelim_data/product_data.csv', index=False)
attribute_merged.to_csv('../../data/prelim_data/attribute_data.csv', index=False)
category_data.to_csv('../../data/prelim_data/category_data.csv', index=False)